# Sınıflandırma Modeli Değerlendirme

Bu ödevde iki farklı sınıflandırma problemi üzerinden model başarı metrikleri hesaplanacaktır:

- **Görev 1:** Churn (müşteri kaybı) tahmini yapan bir modelin olasılık çıktılarından, 0.5 eşik değeri kullanılarak **confusion matrix** oluşturulacak; **Accuracy**, **Precision**, **Recall** ve **F1 Skoru** hesaplanacaktır.
- **Görev 2:** Bir bankanın dolandırıcılık (fraud) tespiti modeline ait hazır confusion matrix üzerinden aynı metrikler hesaplanacak ve modelin neden "başarısız" olarak değerlendirildiği yorumlanacaktır.

Tüm hesaplamalar önce **formüller kullanılarak elle (manuel)**, ardından **scikit-learn** fonksiyonlarıyla doğrulanarak yapılacaktır.

In [1]:
# Sayısal işlemler için NumPy kütüphanesini içe aktar
import numpy as np

# Tablo (DataFrame) işlemleri için Pandas kütüphanesini içe aktar
import pandas as pd

# Hazır confusion matrix ve metrik fonksiyonları için sklearn içe aktarılır
# confusion_matrix   : TP / TN / FP / FN sayılarını üreten karmaşıklık matriksi
# accuracy_score     : (TP + TN) / Toplam -> genel doğruluk oranı
# precision_score    : TP / (TP + FP) -> pozitif dediklerinin ne kadarı gerçekten pozitif
# recall_score       : TP / (TP + FN) -> gerçek pozitiflerin ne kadarını yakalayabildik
# f1_score           : precision ve recall'un harmonik ortalaması
# classification_report : tüm metrikleri tek seferde özetleyen rapor
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

## Görev 1 : Churn Modeli için Confusion Matrix ve Metrik Hesaplama

Müşterinin churn olup olmama durumunu tahminleyen bir sınıflandırma modeli oluşturulmuştur. 10 test verisi gözleminin gerçek değerleri ve modelin tahmin ettiği olasılık değerleri aşağıda verilmiştir.

**Yapılacaklar:**
- Eşik değerini **0.5** alarak confusion matrix oluşturunuz.
- **Accuracy**, **Recall**, **Precision**, **F1** skorlarını hesaplayınız.

| Gözlem | Gerçek Değer | Model Olasılık Tahmini (1 sınıfına ait olma olasılığı) |
|:---:|:---:|:---:|
| 1  | 1 | 0.70 |
| 2  | 1 | 0.80 |
| 3  | 1 | 0.65 |
| 4  | 1 | 0.90 |
| 5  | 1 | 0.45 |
| 6  | 1 | 0.50 |
| 7  | 0 | 0.55 |
| 8  | 0 | 0.35 |
| 9  | 0 | 0.40 |
| 10 | 0 | 0.25 |

In [2]:
# PDF'te verilen 10 gözlemlik test verisini DataFrame olarak oluştur
df1 = pd.DataFrame({
    "gercek_deger": [1, 1, 1, 1, 1, 1, 0, 0, 0, 0],                              # Gerçek sınıf etiketi (churn=1, non-churn=0)
    "olasilik_tahmini": [0.70, 0.80, 0.65, 0.90, 0.45, 0.50, 0.55, 0.35, 0.40, 0.25],  # Modelin "1" sınıfına verdiği olasılık
})

# Gözlem numaralarını 1'den başlatmak için index'i düzenle (PDF'teki sıralamayla eşleşsin diye)
df1.index = range(1, len(df1) + 1)

# Oluşturulan veri setini ekrana yazdır
df1

,gercek_deger,olasilik_tahmini
1,1,0.70
2,1,0.80
3,1,0.65
4,1,0.90
5,1,0.45
6,1,0.50
7,0,0.55
8,0,0.35
9,0,0.40
10,0,0.25


### Adım 1 — Olasılık Tahminlerini Sınıf Etiketine Çevirme (Eşikleme)

Modelin ürettiği olasılık değeri tek başına bir sınıf tahmini değildir; bir **eşik (threshold)** değeri belirlenip bu eşiğin üzerinde/altında kalan olasılıklar sınıf etiketine dönüştürülür.

$$
\hat{y}_i =
\begin{cases}
1, & p_i \geq 0.5 \\
0, & p_i < 0.5
\end{cases}
$$

Burada $p_i$, $i$. gözlemin "1" sınıfına (churn) ait olma olasılığıdır.

In [3]:
# Sorulan eşik değerini değişkende tut (ödevde 0.5 olarak isteniyor)
esik_degeri = 0.5

# Olasılık eşik değerinden büyük veya eşitse 1 (churn), değilse 0 (non-churn) olarak etiketle
# NOT: 6. gözlemde olasılık tam olarak eşik değerine (0.50) eşit olduğu için ">=" kullanılarak
# bu gözlem de pozitif (churn) sınıfına dahil edilmiştir.
df1["tahmin"] = (df1["olasilik_tahmini"] >= esik_degeri).astype(int)

# Gerçek değer ile tahmin edilen değeri karşılaştırmalı olarak göster
df1

,gercek_deger,olasilik_tahmini,tahmin
1,1,0.70,1
2,1,0.80,1
3,1,0.65,1
4,1,0.90,1
5,1,0.45,0
6,1,0.50,1
7,0,0.55,1
8,0,0.35,0
9,0,0.40,0
10,0,0.25,0


### Adım 2 — Confusion Matrix (Karmaşıklık Matriksi) Oluşturma

Confusion matrix, gerçek değerler ile tahmin edilen değerlerin kesişimindeki gözlem sayılarını gösterir:

|                        | Tahmin: Churn (1) | Tahmin: Non-Churn (0) |
|------------------------|:------------------:|:-----------------------:|
| **Gerçek: Churn (1)**     | TP (Doğru Pozitif)  | FN (Yanlış Negatif)     |
| **Gerçek: Non-Churn (0)** | FP (Yanlış Pozitif) | TN (Doğru Negatif)      |

- **TP (True Positive):** Gerçekte churn olan ve model tarafından da churn olarak tahmin edilenler.
- **FN (False Negative):** Gerçekte churn olan ama model tarafından non-churn tahmin edilenler.
- **FP (False Positive):** Gerçekte non-churn olan ama model tarafından churn tahmin edilenler.
- **TN (True Negative):** Gerçekte non-churn olan ve model tarafından da non-churn tahmin edilenler.

In [4]:
# sklearn'ün confusion_matrix fonksiyonu ile karmaşıklık matriksini hesapla
# labels=[1, 0] parametresi ile pozitif sınıf (1=Churn) matriksin sol/üst tarafında yer alır
cm1 = confusion_matrix(df1["gercek_deger"], df1["tahmin"], labels=[1, 0])

# Sayısal matriksi okunabilir bir DataFrame'e dönüştür
cm1_df = pd.DataFrame(
    cm1,
    index=["Gerçek: Churn (1)", "Gerçek: Non-Churn (0)"],
    columns=["Tahmin: Churn (1)", "Tahmin: Non-Churn (0)"],
)

# Confusion matrix'i ekrana yazdır
cm1_df

,Tahmin: Churn (1),Tahmin: Non-Churn (0)
Gerçek: Churn (1),5,1
Gerçek: Non-Churn (0),1,3


In [5]:
# Confusion matrix'ten TP, FN, FP, TN değerlerini tek tek ayıkla
# .loc ile satır/sütun etiketlerine göre ilgili hücreye erişilir
tp1 = cm1_df.loc["Gerçek: Churn (1)", "Tahmin: Churn (1)"]        # Doğru pozitif
fn1 = cm1_df.loc["Gerçek: Churn (1)", "Tahmin: Non-Churn (0)"]    # Yanlış negatif
fp1 = cm1_df.loc["Gerçek: Non-Churn (0)", "Tahmin: Churn (1)"]    # Yanlış pozitif
tn1 = cm1_df.loc["Gerçek: Non-Churn (0)", "Tahmin: Non-Churn (0)"]  # Doğru negatif

# Ayıklanan değerleri kontrol amaçlı yazdır
print(f"TP (Doğru Pozitif)  : {tp1}")
print(f"FN (Yanlış Negatif) : {fn1}")
print(f"FP (Yanlış Pozitif) : {fp1}")
print(f"TN (Doğru Negatif)  : {tn1}")

TP (Doğru Pozitif)  : 5
FN (Yanlış Negatif) : 1
FP (Yanlış Pozitif) : 1
TN (Doğru Negatif)  : 3


### Adım 3 — Accuracy, Precision, Recall, F1 Skorlarının Hesaplanması

$$
Accuracy = \frac{TP + TN}{TP + TN + FP + FN}
$$

$$
Precision = \frac{TP}{TP + FP}
$$

$$
Recall = \frac{TP}{TP + FN}
$$

$$
F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}
$$

In [6]:
# Toplam gözlem sayısı (n = TP + TN + FP + FN)
n1 = tp1 + tn1 + fp1 + fn1

# Accuracy: tüm tahminler içinde doğru bilinenlerin oranı
accuracy1_manuel = (tp1 + tn1) / n1

# Precision: "churn" dediklerimizin ne kadarı gerçekten churn
precision1_manuel = tp1 / (tp1 + fp1)

# Recall (Sensitivity/TPR): gerçekte churn olanların ne kadarını yakalayabildik
recall1_manuel = tp1 / (tp1 + fn1)

# F1 Skoru: precision ve recall'un harmonik ortalaması (ikisini dengeli özetler)
f1_1_manuel = 2 * (precision1_manuel * recall1_manuel) / (precision1_manuel + recall1_manuel)

# Manuel hesaplanan metrikleri yazdır
print("----- Manuel Hesaplama (Görev 1) -----")
print(f"Accuracy  : {accuracy1_manuel:.4f}")
print(f"Precision : {precision1_manuel:.4f}")
print(f"Recall    : {recall1_manuel:.4f}")
print(f"F1 Score  : {f1_1_manuel:.4f}")

----- Manuel Hesaplama (Görev 1) -----
Accuracy  : 0.8000
Precision : 0.8333
Recall    : 0.8333
F1 Score  : 0.8333


In [7]:
# Manuel hesaplamayı sklearn'ün hazır fonksiyonlarıyla doğrula
accuracy1_sk = accuracy_score(df1["gercek_deger"], df1["tahmin"])
precision1_sk = precision_score(df1["gercek_deger"], df1["tahmin"])
recall1_sk = recall_score(df1["gercek_deger"], df1["tahmin"])
f1_1_sk = f1_score(df1["gercek_deger"], df1["tahmin"])

print("----- sklearn ile Doğrulama (Görev 1) -----")
print(f"Accuracy  : {accuracy1_sk:.4f}")
print(f"Precision : {precision1_sk:.4f}")
print(f"Recall    : {recall1_sk:.4f}")
print(f"F1 Score  : {f1_1_sk:.4f}")

# classification_report ile tüm metrikleri (her iki sınıf için de) tek seferde görüntüle
print("\n----- classification_report -----")
print(classification_report(df1["gercek_deger"], df1["tahmin"], target_names=["Non-Churn (0)", "Churn (1)"]))

----- sklearn ile Doğrulama (Görev 1) -----
Accuracy  : 0.8000
Precision : 0.8333
Recall    : 0.8333
F1 Score  : 0.8333

----- classification_report -----
               precision    recall  f1-score   support

Non-Churn (0)       0.75      0.75      0.75         4
    Churn (1)       0.83      0.83      0.83         6

     accuracy                           0.80        10
    macro avg       0.79      0.79      0.79        10
 weighted avg       0.80      0.80      0.80        10



### Görev 1 — Sonuç Özeti

| Metrik | Değer |
|--------|:-----:|
| TP / FN / FP / TN | 5 / 1 / 1 / 3 |
| Accuracy | 0.80 |
| Precision | 0.833 |
| Recall | 0.833 |
| F1 Score | 0.833 |

**Yorum:**
- Model, 10 gözlemin **8 tanesini** doğru sınıflandırmıştır (Accuracy = %80).
- **5. gözlem** (gerçek=churn, olasılık=0.45) eşik değerinin altında kaldığı için yanlışlıkla non-churn tahmin edilmiş (FN); bu, kaybedilebilecek bir müşterinin gözden kaçırılması anlamına gelir.
- **7. gözlem** (gerçek=non-churn, olasılık=0.55) eşik değerinin üzerinde kaldığı için yanlışlıkla churn tahmin edilmiş (FP).
- Precision ve Recall'un birbirine eşit ve makul seviyede (%83.3) olması, modelin pozitif sınıfı (churn) tahmin ederken **dengeli** bir performans sergilediğini gösterir; ne çok fazla yanlış alarm (FP) ne de çok fazla kaçırılan churn (FN) vardır.

## Görev 2 : Fraud (Dolandırıcılık) Modeli için Confusion Matrix Yorumlama

Banka üzerinden yapılan işlemler sırasında dolandırıcılık işlemlerinin yakalanması amacıyla bir sınıflandırma modeli oluşturulmuştur. **%90.5** doğruluk oranı elde edilen modelin başarısı yeterli bulunup model canlıya alınmıştır. Ancak canlıya alındıktan sonra modelin çıktıları beklendiği gibi olmamış, iş birimi modelin başarısız olduğunu iletmiştir.

Modelin tahmin sonuçlarının karmaşıklık matriksi aşağıda verilmiştir:

|                          | Tahmin: Fraud (1) | Tahmin: Non-Fraud (0) | Toplam |
|--------------------------|:------------------:|:-----------------------:|:------:|
| **Gerçek: Fraud (1)**     | 5                  | 5                        | 10     |
| **Gerçek: Non-Fraud (0)** | 90                 | 900                      | 990    |
| **Toplam**                | 95                 | 905                      | 1000   |

**Yapılacaklar:**
- Accuracy, Recall, Precision, F1 skorlarını hesaplayınız.
- Veri Bilimi ekibinin gözden kaçırdığı durum ne olabilir yorumlayınız.

In [8]:
# Bu görevde ham gözlemler değil, doğrudan hazır bir confusion matrix verilmiştir.
# Bu nedenle matriks değerlerini elle tanımlıyoruz.
tp2 = 5    # Gerçek fraud, model de fraud demiş (doğru pozitif)
fn2 = 5    # Gerçek fraud, model non-fraud demiş (kaçırılan dolandırıcılık!)
fp2 = 90   # Gerçek non-fraud, model fraud demiş (yanlış alarm)
tn2 = 900  # Gerçek non-fraud, model de non-fraud demiş (doğru negatif)

# Confusion matrix'i DataFrame olarak görselleştir (PDF'teki tabloyla birebir aynı)
cm2_df = pd.DataFrame(
    [[tp2, fn2],
     [fp2, tn2]],
    index=["Gerçek: Fraud (1)", "Gerçek: Non-Fraud (0)"],
    columns=["Tahmin: Fraud (1)", "Tahmin: Non-Fraud (0)"],
)

# Satır ve sütun toplamlarını da ekleyerek PDF'teki tabloyu tam olarak yeniden oluştur
cm2_df["Toplam"] = cm2_df.sum(axis=1)
cm2_df.loc["Toplam"] = cm2_df.sum()

cm2_df

,Tahmin: Fraud (1),Tahmin: Non-Fraud (0),Toplam
Gerçek: Fraud (1),5,5,10
Gerçek: Non-Fraud (0),90,900,990
Toplam,95,905,1000


In [9]:
# Toplam gözlem sayısı
n2 = tp2 + tn2 + fp2 + fn2

# Accuracy: doğru tahminlerin toplam gözleme oranı
accuracy2_manuel = (tp2 + tn2) / n2

# Precision: "fraud" dediklerimizin ne kadarı gerçekten fraud
precision2_manuel = tp2 / (tp2 + fp2)

# Recall: gerçek fraud işlemlerinin ne kadarını yakalayabildik
recall2_manuel = tp2 / (tp2 + fn2)

# F1 Skoru: precision ve recall'un harmonik ortalaması
f1_2_manuel = 2 * (precision2_manuel * recall2_manuel) / (precision2_manuel + recall2_manuel)

print("----- Manuel Hesaplama (Görev 2) -----")
print(f"Accuracy  : {accuracy2_manuel:.4f}  ({accuracy2_manuel:.1%})")
print(f"Precision : {precision2_manuel:.4f}  ({precision2_manuel:.1%})")
print(f"Recall    : {recall2_manuel:.4f}  ({recall2_manuel:.1%})")
print(f"F1 Score  : {f1_2_manuel:.4f}  ({f1_2_manuel:.1%})")

----- Manuel Hesaplama (Görev 2) -----
Accuracy  : 0.9050  (90.5%)
Precision : 0.0526  (5.3%)
Recall    : 0.5000  (50.0%)
F1 Score  : 0.0952  (9.5%)


### sklearn ile Doğrulama

Elimizde ham gözlemler yok, sadece hazır confusion matrix var. Yine de doğrulama yapabilmek için, bu matriksle **birebir aynı confusion matrix'i üretecek** şekilde 1000 elemanlık bir `gerçek`/`tahmin` dizisi kurgulayıp sklearn fonksiyonlarını buna uygulayabiliriz.

In [10]:
# Confusion matrix'teki TP/FN/FP/TN sayılarına birebir uyacak şekilde
# 1000 elemanlık gerçek değer ve tahmin dizilerini yeniden kurgula.
# Sıralama: önce TP kadar (1,1), sonra FN kadar (1,0), sonra FP kadar (0,1), sonra TN kadar (0,0)
gercek2 = np.array([1] * tp2 + [1] * fn2 + [0] * fp2 + [0] * tn2)
tahmin2 = np.array([1] * tp2 + [0] * fn2 + [1] * fp2 + [0] * tn2)

# Kurgulanan diziden sklearn ile confusion matrix üretip orijinal tabloyla karşılaştır
cm2_check = confusion_matrix(gercek2, tahmin2, labels=[1, 0])
print("sklearn confusion_matrix çıktısı ([[TP, FN], [FP, TN]] formatında):")
print(cm2_check)

# sklearn fonksiyonlarıyla metrikleri tekrar hesapla
accuracy2_sk = accuracy_score(gercek2, tahmin2)
precision2_sk = precision_score(gercek2, tahmin2)
recall2_sk = recall_score(gercek2, tahmin2)
f1_2_sk = f1_score(gercek2, tahmin2)

print("\n----- sklearn ile Doğrulama (Görev 2) -----")
print(f"Accuracy  : {accuracy2_sk:.4f}")
print(f"Precision : {precision2_sk:.4f}")
print(f"Recall    : {recall2_sk:.4f}")
print(f"F1 Score  : {f1_2_sk:.4f}")

# classification_report ile detaylı özet
print("\n----- classification_report -----")
print(classification_report(gercek2, tahmin2, target_names=["Non-Fraud (0)", "Fraud (1)"]))

sklearn confusion_matrix çıktısı ([[TP, FN], [FP, TN]] formatında):
[[  5   5]
 [ 90 900]]

----- sklearn ile Doğrulama (Görev 2) -----
Accuracy  : 0.9050
Precision : 0.0526
Recall    : 0.5000
F1 Score  : 0.0952

----- classification_report -----
               precision    recall  f1-score   support

Non-Fraud (0)       0.99      0.91      0.95       990
    Fraud (1)       0.05      0.50      0.10        10

     accuracy                           0.91      1000
    macro avg       0.52      0.70      0.52      1000
 weighted avg       0.99      0.91      0.94      1000



### Görev 2 : Sonuç Özeti ve Yorum

| Metrik | Değer |
|--------|:-----:|
| TP / FN / FP / TN | 5 / 5 / 90 / 900 |
| Accuracy | 0.905 (%90.5) |
| Precision | 0.0526 (%5.26) |
| Recall | 0.50 (%50) |
| F1 Score | 0.0952 (%9.52) |

**Veri Bilimi ekibinin gözden kaçırdığı durum:**

1. **Sınıf dengesizliği (class imbalance) göz ardı edilmiştir.** Veri setindeki 1000 işlemin sadece 10 tanesi (%1) fraud'dur; geri kalan 990 işlem (%99) non-fraud'dur. Bu kadar dengesiz bir veri setinde **Accuracy yanıltıcı bir metriktir**. Nitekim hiçbir işlemi incelemeden, tüm işlemleri "non-fraud" olarak etiketleyen anlamsız/naif bir model bile 990/1000 = **%99 accuracy** elde ederdi. Dolayısıyla modelin elde ettiği %90.5'lik accuracy, aslında bu naif modelin bile gerisinde kalan, gerçekte kötü bir performanstır.

2. **Asıl kritik metrikler (Precision ve Recall) incelenmemiştir.**
   - **Recall = %50:** Model, gerçekte var olan 10 dolandırıcılık işleminin sadece **5 tanesini** yakalayabilmiş; diğer **5 tanesi (FN)** hiç fark edilmeden sisteme geçmiştir. Bu, bankanın doğrudan finansal kayıp yaşayacağı bir durumdur.
   - **Precision = %5.26:** Model "fraud" dediği 95 işlemden sadece **5 tanesi** gerçekten fraud'dur; **90 tanesi (FP)** aslında normal (masum) işlemlerdir. Bu, çok sayıda müşterinin işleminin gereksiz yere durdurulması/incelemeye alınması anlamına gelir ve ciddi bir **müşteri deneyimi ve operasyonel maliyet** sorunu yaratır.
   - **F1 Skoru = %9.52** gibi çok düşük bir değerde olması, precision ve recall'un ikisinin de zayıf olduğunun bir göstergesidir.

3. **Sonuç:** İş biriminin "model başarısız" tepkisi haklıdır. Model canlıya alınmadan önce, dengesiz veri setlerinde accuracy yerine **Precision, Recall, F1 Skoru** ve mümkünse **PR-AUC (Precision-Recall Curve)** gibi metriklerle değerlendirilmeli; ayrıca farklı eşik değerleri denenerek recall-precision dengesi iş ihtiyacına göre (örn. dolandırıcılığı kaçırmamak önceliğiyle recall'u artıracak şekilde) ayarlanmalıydı.

## Genel Sonuç Karşılaştırması

| Metrik | Görev 1 (Churn) | Görev 2 (Fraud) |
|--------|:---------------:|:----------------:|
| Accuracy | 0.800 | 0.905 |
| Precision | 0.833 | 0.053 |
| Recall | 0.833 | 0.500 |
| F1 Score | 0.833 | 0.095 |

**Genel Yorum:** İki görev, tek bir metriğe (özellikle Accuracy'ye) güvenmenin ne kadar yanıltıcı olabileceğini net bir şekilde ortaya koymaktadır. Görev 2'deki model, Görev 1'deki modelden **daha yüksek accuracy** değerine sahip olmasına rağmen, asıl amaç olan dolandırıcılığı yakalama konusunda **çok daha başarısızdır**. Bu nedenle model değerlendirmesi, veri setinin sınıf dağılımı ve iş probleminin önceliği (örn. false negative'in mi yoksa false positive'in mi daha maliyetli olduğu) göz önünde bulundurularak, **birden fazla metrik bir arada** yorumlanarak yapılmalıdır.